In [62]:
from __future__ import annotations

import importlib
import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.colors import LogNorm

pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option('display.max_rows', 500)


In [2]:
def run_query(sql: str) -> pd.DataFrame:
    config_dir = Path.home() / '.pulsepoint'
    sys.path.insert(0, str(config_dir))

    from dotenv import load_dotenv

    load_dotenv(config_dir / '.env', override=False)
    dbfuncs = importlib.import_module('dbfuncs')
    return dbfuncs.query2df(sql, print_time=True)

In [67]:
run_query("""
select distinct * from iceberg.thirdparty.pii_clean  
where npi='1639530496'
and day='2026-09-28'
order by source, npi, state, profession, specialties
""")

Query executed in 20.022 seconds


,day,source,id,first_name,last_name,addr_1,addr_2,city,state,zip,phone,npi,profession,specialties,emails,hashedemails,datavendor,accountid
0,2026-09-28,medscape,2e7813af237f027af9d042e63603eca225aa4d514cfed629dbcd9c95b8d5c5fc,ANDREA,HUNT,635 NW 88th DRive,,,FL,33069,,1639530496,Nurse/Advanced Practice Nurse,Administration/Management,ahuntguelph@hotmail.com,,MEDSCAPE REGISTRATION,MS
1,2026-09-28,medscape,2e7813af237f027af9d042e63603eca225aa4d514cfed629dbcd9c95b8d5c5fc,ANDREA,HUNT,635 NW 88th DRive,,,FL,33069,,1639530496,Nurse/Advanced Practice Nurse,Diabetes,ahuntguelph@hotmail.com,,MEDSCAPE REGISTRATION,MS
2,2026-09-28,medscape,2e7813af237f027af9d042e63603eca225aa4d514cfed629dbcd9c95b8d5c5fc,ANDREA,HUNT,635 NW 88th DRive,,,FL,33069,,1639530496,Nurse/Advanced Practice Nurse,Geriatrics,ahuntguelph@hotmail.com,,MEDSCAPE REGISTRATION,MS
3,2026-09-28,medscape,2e7813af237f027af9d042e63603eca225aa4d514cfed629dbcd9c95b8d5c5fc,ANDREA,HUNT,,,,MO,63401,,1639530496,Nurse/Advanced Practice Nurse,Administration/Management,iamlabocagrande@gmail.com,,MEDSCAPE REGISTRATION,MS
4,2026-09-28,medscape,2e7813af237f027af9d042e63603eca225aa4d514cfed629dbcd9c95b8d5c5fc,ANDREA,HUNT,,,,MO,63401,,1639530496,Nurse/Advanced Practice Nurse,Diabetes,iamlabocagrande@gmail.com,,MEDSCAPE REGISTRATION,MS
5,2026-09-28,medscape,2e7813af237f027af9d042e63603eca225aa4d514cfed629dbcd9c95b8d5c5fc,ANDREA,HUNT,,,,MO,63401,,1639530496,Nurse/Advanced Practice Nurse,Geriatrics,iamlabocagrande@gmail.com,,MEDSCAPE REGISTRATION,MS
6,2026-09-28,medscape,2e7813af237f027af9d042e63603eca225aa4d514cfed629dbcd9c95b8d5c5fc,ANDREA,HUNT,,,,OK,73130,,1639530496,Nurse/Advanced Practice Nurse,Administration/Management,andrea.rae.hunt@gmail.com,,MEDSCAPE REGISTRATION,MS
7,2026-09-28,medscape,2e7813af237f027af9d042e63603eca225aa4d514cfed629dbcd9c95b8d5c5fc,ANDREA,HUNT,,,,OK,73130,,1639530496,Nurse/Advanced Practice Nurse,Diabetes,andrea.rae.hunt@gmail.com,,MEDSCAPE REGISTRATION,MS
8,2026-09-28,medscape,2e7813af237f027af9d042e63603eca225aa4d514cfed629dbcd9c95b8d5c5fc,ANDREA,HUNT,,,,OK,73130,,1639530496,Nurse/Advanced Practice Nurse,Geriatrics,andrea.rae.hunt@gmail.com,,MEDSCAPE REGISTRATION,MS
9,2026-09-28,stdvendor,2e7813af237f027af9d042e63603eca225aa4d514cfed629dbcd9c95b8d5c5fc,,,,,,,,,1639530496,,,agreen@netsync.net,,quantummedica,QM


In [44]:
run_query("""
select distinct * from iceberg.onboard.liveintentpii 
where regexp_replace(npi, '^[^_]+_', '') = '1639530496'
and day='2026-09-28'
""")

Query executed in 12.823 seconds


,day,npi,hashedemail
0,2026-09-28,MS_1639530496,8a6797263d39bea884055deacac7b26968ae939f3cf2dbd497efa861e0f1c20c
1,2026-09-28,QM_1639530496,94cc560e8e28f7d65ed3519d8c7ab7e68553bbd69577c2163bf6cdd6fc462091
2,2026-09-28,MS_1639530496,b0ba99ee6cc8546d23d557b1577f7b6a911160d24ced63ec9de584b83f18ad9c
3,2026-09-28,MS_1639530496,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e


In [77]:
run_query("""
select * from iceberg.crossscreen.nscreen_ipcolocation_daily_ua where day='2026-09-28' limit 1
""")

Query executed in 4.436 seconds


,day,uid1,uid2,weight,samedevice,differentusers
0,2026-09-28,8a193352-2c05-5273-8b1b-7a9140d26e18,9d46df8d-44f8-453e-9950-ec2b01a9b037,0.200,False,False


In [81]:
run_query("""select * from iceberg.crossscreen.nscreen_geo_daily_ua 
where day='2026-09-28' and uid in ('part_tUdyCHSiwZte', '2FFJjfAgFAIq', '389dc412-2bd3-4189-9eb4-c0a1e087ccde') """)

Query executed in 8.245 seconds


,day,uid,geo
0,2026-09-28,part_tUdyCHSiwZte,333


In [52]:
run_query(""" 
WITH matching_emails AS (
    SELECT DISTINCT 
        regexp_extract(npi, '([0-9]{10})$',1) as npi, 
        lower(hashedemail) AS hashedemail
    FROM iceberg.onboard.liveintentpii
    WHERE day = '2026-09-28'
      AND regexp_extract(
          npi,
          '([0-9]{10})$',
          1
      ) = '1639530496'
),

matching_idsync AS (
    SELECT idsync.*, emails.npi as npi
    FROM iceberg.thirdparty.liveintent_idsync idsync
    INNER JOIN matching_emails emails
        ON lower(idsync.hashedemail_sha256) = emails.hashedemail
    WHERE idsync.day = '2026-09-28'
),
matching_uids as (
    SELECT DISTINCT
        day,
        npi,
        liid,
        hashedemail_sha256,
        uid_token,
        split_part(uid_token, ':', 1) AS uid_type,
        split_part(uid_token, ':', 2) AS uid
    FROM matching_idsync
    CROSS JOIN UNNEST(
        split(uids, '|')
    ) AS exploded(uid_token)
    WHERE uid_token <> ''
),
vendor_data as (
    select * from iceberg.jteixeira_ipa.nscreen_membership_sample_liveramp_20260928_line_filter_v1_5pct
    union all
    select * from iceberg.jteixeira_ipa.nscreen_membership_sample_experian_20260928_line_filter_v1_5pct
    union all
    select * from iceberg.jteixeira_ipa.nscreen_membership_sample_throtle_20260928_line_filter_v1_5pct
)
select * 
from matching_uids as mu
left join vendor_data as v
    on mu.uid=v.uid
where matchid is not null
""")

Query executed in 17.326 seconds


,day,npi,liid,hashedemail_sha256,uid_token,uid_type,uid,day,vendor,uid,matchid,householdid,stable
0,2026-09-28,1639530496,yDpCHcmoBrMFRuM-wiE25z7xPYjPPG2ir2ewrA,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e,98251:mGSIKVDm4ctO,98251,mGSIKVDm4ctO,2026-09-28,throtle,mGSIKVDm4ctO,-8807694220336044580,NaN,True
1,2026-09-28,1639530496,yDpCHcmoBrMFRuM-wiE25z7xPYjPPG2ir2ewrA,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e,98251:2FFJjfAgFAIq,98251,2FFJjfAgFAIq,2026-09-28,throtle,2FFJjfAgFAIq,-8807694220336044580,NaN,True
2,2026-09-28,1639530496,yDpCHcmoBrMFRuM-wiE25z7xPYjPPG2ir2ewrA,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e,98251:2FFJjfAgFAIq,98251,2FFJjfAgFAIq,2026-09-28,liveramp,2FFJjfAgFAIq,3597744279521006960,8823308120729131008.000,True
3,2026-09-28,1639530496,yDpCHcmoBrMFRuM-wiE25z7xPYjPPG2ir2ewrA,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e,aaid:14a008b9-4e91-36ea-a873-49f7b5497194,aaid,14a008b9-4e91-36ea-a873-49f7b5497194,2026-09-28,throtle,14a008b9-4e91-36ea-a873-49f7b5497194,-8807694220336044580,NaN,True
4,2026-09-28,1639530496,yDpCHcmoBrMFRuM-wiE25z7xPYjPPG2ir2ewrA,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e,98251:4eAokXU4vO9g,98251,4eAokXU4vO9g,2026-09-28,throtle,4eAokXU4vO9g,-8807694220336044580,NaN,True
5,2026-09-28,1639530496,yDpCHcmoBrMFRuM-wiE25z7xPYjPPG2ir2ewrA,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e,98251:part_Y9joaNRMPsUr,98251,part_Y9joaNRMPsUr,2026-09-28,throtle,part_Y9joaNRMPsUr,-8807694220336044580,NaN,True
6,2026-09-28,1639530496,yDpCHcmoBrMFRuM-wiE25z7xPYjPPG2ir2ewrA,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e,98251:part_tUdyCHSiwZte,98251,part_tUdyCHSiwZte,2026-09-28,throtle,part_tUdyCHSiwZte,-8807694220336044580,NaN,True
7,2026-09-28,1639530496,yDpCHcmoBrMFRuM-wiE25z7xPYjPPG2ir2ewrA,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e,98251:part_tUdyCHSiwZte,98251,part_tUdyCHSiwZte,2026-09-28,experian,part_tUdyCHSiwZte,-5391460821255451246,-6235083072056879104.000,True
8,2026-09-28,1639530496,yDpCHcmoBrMFRuM-wiE25z7xPYjPPG2ir2ewrA,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e,98251:part_tUdyCHSiwZte,98251,part_tUdyCHSiwZte,2026-09-28,liveramp,part_tUdyCHSiwZte,3597744279521006960,8823308120729131008.000,True
9,2026-09-28,1639530496,yDpCHcmoBrMFRuM-wiE25z7xPYjPPG2ir2ewrA,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e,aaid:389dc412-2bd3-4189-9eb4-c0a1e087ccde,aaid,389dc412-2bd3-4189-9eb4-c0a1e087ccde,2026-09-28,liveramp,389dc412-2bd3-4189-9eb4-c0a1e087ccde,3597744279521006960,8823308120729131008.000,True


In [ ]:
test_matchids=run_query("""
WITH matching_emails AS (
    SELECT DISTINCT 
        regexp_extract(npi, '^([^_]+)_', 1) as email_source, 
        regexp_extract(npi, '([0-9]{10})$',1) as npi, 
        lower(hashedemail) AS hashedemail
    FROM iceberg.onboard.liveintentpii
    WHERE day = '2026-09-28'
),
matching_pii AS (
    SELECT DISTINCT me.npi, me.email_source, me.hashedemail, first_name, last_name,  state, zip, accountid as pii_source, 
    lower(to_hex(sha256(to_utf8(lower(trim(emails)))))) as pii_hashedemail
    FROM matching_emails me 
    left join iceberg.thirdparty.pii_clean pc
    on me.npi = pc.npi 
        and me.hashedemail=lower(to_hex(sha256(to_utf8(lower(trim(emails))))))
    and pc.day = '2026-09-28'
),
matching_idsync AS (
    SELECT idsync.*, mp.email_source, mp.npi as npi, pii_source, first_name, last_name, state, zip, pii_hashedemail
    FROM iceberg.thirdparty.liveintent_idsync idsync
    INNER JOIN matching_pii mp
        ON lower(idsync.hashedemail_sha256) = mp.hashedemail
    WHERE idsync.day = '2026-09-28'
),
matching_uids as (
    SELECT DISTINCT
        day, email_source,
        npi, pii_source, first_name, last_name, state, zip, pii_hashedemail,
        liid,
        hashedemail_sha256,
        uid_token,
        split_part(uid_token, ':', 1) AS uid_type,
        split_part(uid_token, ':', 2) AS uid
    FROM matching_idsync
    CROSS JOIN UNNEST(
        split(uids, '|')
    ) AS exploded(uid_token)
    WHERE uid_token <> ''
),
vendor_data as (
    select * from iceberg.jteixeira_ipa.nscreen_membership_sample_liveramp_20260928_line_filter_v1_5pct
    union all
    select * from iceberg.jteixeira_ipa.nscreen_membership_sample_experian_20260928_line_filter_v1_5pct
    union all
    select * from iceberg.jteixeira_ipa.nscreen_membership_sample_throtle_20260928_line_filter_v1_5pct
),
geo_daily_ua as (
    select * from iceberg.crossscreen.nscreen_geo_daily_ua 
    where day='2026-09-28' and uid in (select uid from vendor_uid_evidence)
),
vendor_uid_evidence as (
    select distinct v.matchid, v.vendor, v.uid, hashedemail_sha256, email_source, npi, pii_source, first_name, last_name, state, zip, pii_hashedemail
    from vendor_data as v
    left join matching_uids as mu
        on mu.uid=v.uid
    left join geo_daily_ua gdua
        on mu.uid=gdua.uid
),
joined_data as (
    select 
)
select * from vendor_uid_evidence
where matchid in (3597744279521006960, -8807694220336044580, -5391460821255451246)
order by vendor, matchid, uid, npi
""")

display(test_matchids)

Query executed in 22.311 seconds


,matchid,vendor,uid,hashedemail_sha256,email_source,npi,pii_source,first_name,last_name,state,zip,pii_hashedemail
0,-5391460821255451246,experian,EP_3098108377,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,-5391460821255451246,experian,TA_797f5b23-eae7-44aa-8200-49860e363ead,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,-5391460821255451246,experian,TA_8509f4b1-687b-4fb6-859a-ee798a4223dd,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,-5391460821255451246,experian,TA_d5111b88-2c25-4ebe-b02c-84c67cd14bfc,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,-5391460821255451246,experian,TA_f0a61c96-106d-42b2-8ec1-8d83579fc7eb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,-5391460821255451246,experian,part_tUdyCHSiwZte,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e,MS,1639530496,MS,ANDREA,HUNT,FL,33069,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e
6,3597744279521006960,liveramp,2FFJjfAgFAIq,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e,MS,1639530496,MS,ANDREA,HUNT,FL,33069,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e
7,3597744279521006960,liveramp,389dc412-2bd3-4189-9eb4-c0a1e087ccde,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e,MS,1639530496,MS,ANDREA,HUNT,FL,33069,0d3a00d5ff4b660ab0fe03897edd4be8f94a19bf8b4f4e778a561bce9270012e
8,3597744279521006960,liveramp,3jzhHSVz08XE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,3597744279521006960,liveramp,4aef9d13-8a51-4ea5-a8fd-e305e9521b5d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
test_matchids=run_query("""
WITH matching_emails AS (
    SELECT DISTINCT 
        regexp_extract(npi, '^([^_]+)_', 1) as email_source, 
        regexp_extract(npi, '([0-9]{10})$',1) as npi, 
        lower(hashedemail) AS hashedemail
    FROM iceberg.onboard.liveintentpii
    WHERE day = '2026-09-28'
),
matching_pii AS (
    SELECT DISTINCT me.npi, me.email_source, me.hashedemail, first_name, last_name,  state, zip, accountid as pii_source, 
    lower(to_hex(sha256(to_utf8(lower(trim(emails)))))) as pii_hashedemail
    FROM matching_emails me 
    left join iceberg.thirdparty.pii_clean pc
    on me.npi = pc.npi 
        and me.hashedemail=lower(to_hex(sha256(to_utf8(lower(trim(emails))))))
    and pc.day = '2026-09-28'
),
matching_idsync AS (
    SELECT idsync.*, mp.email_source, mp.npi as npi, pii_source, first_name, last_name, state, zip, pii_hashedemail
    FROM iceberg.thirdparty.liveintent_idsync idsync
    INNER JOIN matching_pii mp
        ON lower(idsync.hashedemail_sha256) = mp.hashedemail
    WHERE idsync.day = '2026-09-28'
),
matching_uids as (
    SELECT DISTINCT
        day, email_source,
        npi, pii_source, first_name, last_name, state, zip, pii_hashedemail,
        liid,
        hashedemail_sha256,
        uid_token,
        split_part(uid_token, ':', 1) AS uid_type,
        split_part(uid_token, ':', 2) AS uid
    FROM matching_idsync
    CROSS JOIN UNNEST(
        split(uids, '|')
    ) AS exploded(uid_token)
    WHERE uid_token <> ''
),
vendor_data as (
    select * from iceberg.jteixeira_ipa.nscreen_membership_sample_liveramp_20260928_line_filter_v1_5pct
    union all
    select * from iceberg.jteixeira_ipa.nscreen_membership_sample_experian_20260928_line_filter_v1_5pct
    union all
    select * from iceberg.jteixeira_ipa.nscreen_membership_sample_throtle_20260928_line_filter_v1_5pct
),
vendor_uid_evidence as (
    select distinct v.matchid, v.vendor, v.uid, hashedemail_sha256, email_source, npi, pii_source, first_name, last_name, state, zip, pii_hashedemail
    from vendor_data as v
    left join matching_uids as mu
        on mu.uid=v.uid
)
select * from vendor_uid_evidence
where matchid in (3597744279521006960, -8807694220336044580, -5391460821255451246)
order by vendor, matchid, uid, npi
""")

display(test_matchids)